In this notebook, we will show how a full relevance function can be specified and controlled in a search engine (```Apache Solr```).

The BM25 algorithm is the default similarity function in Apache Lucene, Apache Solr, OpenSearch, Elasticsearch, and other Lucene-based search engines: BM25

### BM25 (Best Match Okapi 25)
The algorithm leverages TF and IDF, but also incorporates many additional configurable options.

In [1]:
import math
from collections import Counter

class BM25:
    """Function to calculate BM25 score"""
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.doc_lengths = [len(doc) for doc in corpus]
        self.avg_doc_len = sum(self.doc_lengths) / self.corpus_size
        self.doc_term_frequencies = [Counter(doc) for doc in corpus]

        # Calculate IDF
        self.doc_frequencies = Counter()
        for doc in corpus: self.doc_frequencies.update(set(doc))
        self.idf = {term: math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1) for term, freq in self.doc_frequencies.items()}

    def search(self, query_tokens, top_n=5):
        """Function to search for the top N documents based on BM25 score"""
        scores = []
        for i, doc in enumerate(self.doc_term_frequencies):
            score = 0.0
            for term in query_tokens:
                if term in self.idf:
                    tf = doc[term]
                    numerator = tf * (self.k1 + 1)
                    denominator = tf + self.k1 * (1 - self.b + self.b * (self.doc_lengths[i] / self.avg_doc_len))
                    score += self.idf[term] * (numerator / denominator)
            scores.append((score, i))
        return sorted(scores, key=lambda x: x[0], reverse=True)[:top_n]

### Key Parameters
- `k1` (1.2–2.0): Controls term frequency saturation. 
- `b` (0.75): Controls document length normalization.

In [2]:
docs = [
    {
        "id": "doc1",
        "title": "Worst",
        "description": "The interesting thing is that the person in the wrong made the right decision in the end."
    },
    {
        "id": "doc2",
        "title": "Best",
        "description": "My favorite book is the cat in the hat, which is about a crazy cat who breaks into a house and creates a crazy afternoon for two kids."
        
    },
    {
        "id": "doc3",
        "title": "Okay",
        "description": "My neighbors let the stray cat stay in their garage, which resulted in my favorite hat that I let them borrow being ruined."        
    }
]

In [3]:
query = "the cat in the hat"

### Executing the Search

In [4]:
# Prepare corpus by extracting and tokenizing the descriptions of the documents
corpus = [doc["description"].lower().split() for doc in docs]

In [5]:
bm25 = BM25(corpus)

In [6]:
query_tokens = query.lower().split()

In [7]:
results = bm25.search(query_tokens, top_n=3)

In [8]:
# 5. Display the results clearly
print(f"Results for query: '{query}'\n" + "-"*40)

for score, doc_idx in results:
    matched_doc = docs[doc_idx]
    print(f"Score: {score:.4f}")
    print(f"ID:    {matched_doc['id']}")
    print(f"Title: {matched_doc['title']}")
    print(f"Desc:  {matched_doc['description']}")
    print("-" * 40)

Results for query: 'the cat in the hat'
----------------------------------------
Score: 1.8965
ID:    doc3
Title: Okay
Desc:  My neighbors let the stray cat stay in their garage, which resulted in my favorite hat that I let them borrow being ruined.
----------------------------------------
Score: 1.0997
ID:    doc2
Title: Best
Desc:  My favorite book is the cat in the hat, which is about a crazy cat who breaks into a house and creates a crazy afternoon for two kids.
----------------------------------------
Score: 0.7442
ID:    doc1
Title: Worst
Desc:  The interesting thing is that the person in the wrong made the right decision in the end.
----------------------------------------


### NOTE
- Read about how `Solr` helps with multiple configs.
- Matching vs. Ranking
- Separating Concerns: Filtering vs. Scoring
- Multiplicative vs. Additive Boosting
